# Consigna del desafío 2

**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado**

Recuerden que su notebook de entrega debe poder correrse de inicio a fin sin la aparición de errores.

- Crear sus propios vectores con Gensim basado en lo visto en clase con un corpus propio (revisar enlaces sugeridos en clase 2 sobre opciones de dataset)
- Elegir términos de interés y buscar términos más similares y menos similares.
- Realizar una reduccion de dimensionalidad a los embeddings, llevándolos a 2 dimensiones. Graficar los embeddings proyectados y seleccionar una cantidad de términos (variable MAX_WORDS) de forma tal que la visualización sea adecuada.
- Inspeccionar el grafico y buscar pequeños grupos de palabras que puedan formarse. Interpretarlos e intentar obtener conclusiones. En lo posible, acompañar los grupos de palabras con capturas (y pegarlas en celdas de texto)

##### Resolución
Se decide usar algunos libros de Fyodor Dostoyevsky (en inglés) como corpus, descargados de [Project Gutenberg](https://www.gutenberg.org/).

Estos libros contienen un encabezado y un pie, con metadata, propios de Gutenberg, pero se indica claramente donde empieza el libro y donde termina, mediante el texto `*** START OF ... ***` y `*** END OF ... ***`, por lo cual se utiliza regex para encontrar estas cadenas y mantener únicamente lo que se encuentra entre ellas.

Se unen todos en una misma variable de texto a la cual se aplicará el preprocesamiento.

In [1]:
import os
import urllib.request
import re

BOOKS_DIR = "books_dataset"
os.makedirs(BOOKS_DIR, exist_ok=True)

books = {
    "crime_and_punishment.txt": "https://www.gutenberg.org/files/2554/2554-0.txt",
    "the_brothers_karamazov.txt": "https://www.gutenberg.org/files/28054/28054-0.txt",
    "white_nights_and_other_stories.txt": "https://www.gutenberg.org/files/36034/36034-8.txt",
    "the_idiot.txt": "https://www.gutenberg.org/files/2638/2638-0.txt",
}

In [2]:
for filename, url in books.items():
    filepath = os.path.join(BOOKS_DIR, filename)
    if not os.path.exists(filepath):
        print(f"Descargando {filename}...")
        urllib.request.urlretrieve(url, filepath)
        print("Hecho")
    else:
        print(f"{filename} ya existe, se omite la descarga")

crime_and_punishment.txt ya existe, se omite la descarga
the_brothers_karamazov.txt ya existe, se omite la descarga
white_nights_and_other_stories.txt ya existe, se omite la descarga
the_idiot.txt ya existe, se omite la descarga


In [3]:
def extract_gutenberg_text(raw):
    match = re.search(r'\*\*\* START OF .+? \*\*\*(.+?)\*\*\* END OF .+? \*\*\*', raw, re.DOTALL)
    if match:
        return match.group(1).strip()
    else:
        print("No se encontraron las marcas de Gutenberg. Se retorna archivo completo")
        return raw

In [49]:
# Se compone un único texto con el contenido de todos los libros.
all_text = ""
for book in os.listdir(BOOKS_DIR):
    with open(os.path.join(BOOKS_DIR, book), "r", encoding="utf-8", errors="ignore") as f:
        raw = f.read()
        all_text += "\n"
        all_text += extract_gutenberg_text(raw)
        print(f"Agregando {book}... ({len(all_text)})")


Agregando crime_and_punishment.txt... (1135109)
Agregando the_brothers_karamazov.txt... (3070318)
Agregando the_idiot.txt... (4418014)
Agregando white_nights_and_other_stories.txt... (5068452)


In [ ]:
# Preprocesamiento del texto:
# - Se convierte todo el texto a minúscula
# - Se separa por oraciones (puntos, saltos de línea, signo de exclamación/interrogación)
# - Se ignoran caracteres que no sean alfanuméricos
# *Se evita usar la función text_to_word_sequence ya que la documentación indica que se
# encuentra deprecada (Todo el módulo tf.keras.preprocessing está deprecado)
# text = all_text.lower()
# sentences = re.split(r'[.\n!?]+', text)
# print(len(sentences))
# sentences = [
#     re.findall(r"[a-z0-9]+", s)
#     for s in sentences
# ]
# print(len(sentences))

135394
135394


In [50]:
# Se evita usar la función text_to_word_sequence ya que la documentación indica que se
# encuentra deprecada (Todo el módulo tf.keras.preprocessing está deprecado)
# Por esto se usa la librería nltk.

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Gabriel\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [51]:
sentence_tokens = []
sentences = sent_tokenize(all_text)  # lista de oraciones (strings)

for s in sentences:
    sentence_tokens.append(word_tokenize(s.lower()))

In [59]:
# A ver cómo quedan las oraciones y palabras
print(sentences[0])
print(sentences[:1])
print(sentence_tokens[0])


CRIME AND PUNISHMENT

By Fyodor Dostoevsky



Translated By Constance Garnett




TRANSLATOR’S PREFACE

A few words about Dostoevsky himself may help the English reader to
understand his work.
['\nCRIME AND PUNISHMENT\n\nBy Fyodor Dostoevsky\n\n\n\nTranslated By Constance Garnett\n\n\n\n\nTRANSLATOR’S PREFACE\n\nA few words about Dostoevsky himself may help the English reader to\nunderstand his work.']
['crime', 'and', 'punishment', 'by', 'fyodor', 'dostoevsky', 'translated', 'by', 'constance', 'garnett', 'translator', '’', 's', 'preface', 'a', 'few', 'words', 'about', 'dostoevsky', 'himself', 'may', 'help', 'the', 'english', 'reader', 'to', 'understand', 'his', 'work', '.']


In [55]:
from gensim.models.callbacks import CallbackAny2Vec
from gensim.models import Word2Vec

In [54]:
class callback(CallbackAny2Vec):
    """
    Callback to print loss after each epoch
    """
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print('Loss after epoch {}: {}'.format(self.epoch, loss))
        else:
            print('Loss after epoch {}: {}'.format(self.epoch, loss- self.loss_previous_step))
        self.epoch += 1
        self.loss_previous_step = loss

In [ ]:
# Para el entrenamiento se considera que el corpus es grande y el vocabulario
# relativamente extenso, esto lleva a los siguientes parámetros del modelo:

w2v_model = Word2Vec(
    vector_size=300,    # Permite relaciones más complejas, aunque requiere muchos datos y cómputo
    window=6,           # Se usa una ventana grande, ya que se considera que el significado puede cambiar bastante con el contexto
    min_count=10,       # Se reduce el ruido que pueda aparecer de Gutenberg
    sg=1,               # Skip-gram suele rendir mejor con vocabulario grande
    negative=20,        # 
    workers=1,          #
)